In [ ]:
# ============================================================
# SECTION 1: INSTALL
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new]', 'trl>=0.8.6', 'transformers>=4.40',
    'accelerate', 'requests', 'matplotlib', 'datasets', 'peft'], check=False)
print('Install done')

In [ ]:
# ============================================================
# SECTION 2: CONSTANTS & AUTH
# ============================================================
import os, json, re, time, requests, random
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SPACE_URL   = 'https://Bharath-1608-social-agent-negotiation-v1.hf.space'
HF_REPO_ID  = 'Bharath-1608/social-agent-negotiation-grpo'
MODEL_ID    = 'unsloth/Meta-Llama-3.1-8B-Instruct'
TIMEOUT     = 30
MAX_TURNS   = 15
TRAIN_TASKS = ['single-round-consensus', 'adversarial-information', 'opioid-overdose']
EPISODES_PER_TASK = 20
N_EPOCHS    = 3
GROUP_SIZE  = 4

VALID_ACTIONS = [
    'share_information', 'propose_consensus', 'accept_consensus',
    'reject_consensus', 'challenge_proposal', 'request_clarification', 'flag_bias',
]

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print('HF login OK')
else:
    print('WARNING: HF_TOKEN not found — Hub push will fail')

In [ ]:
# ============================================================
# SECTION 3: MODEL LOAD (Unsloth 4-bit + LoRA)
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    target_modules = ['q_proj', 'v_proj'],
    lora_dropout   = 0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print('Model ready:', MODEL_ID)

In [ ]:
# ============================================================
# SECTION 4: ROLLOUT HELPERS
# ============================================================

def _format_prompt(obs: dict, agent_id: str) -> str:
    pinfo = json.dumps(obs.get('private_information', {}), indent=2)[:600]
    hist  = obs.get('shared_conversation_history', [])[-4:]
    hist_str = '\n'.join(
        f"  [{m['agent_id']}] {m['action_type']}: {str(m.get('content',''))[:200]}"
        for m in hist
    ) or '  (none yet)'
    avail = ', '.join(obs.get('available_actions', VALID_ACTIONS))
    phase = obs.get('current_phase', 'triage')
    pturn = obs.get('phase_turn', 0)
    agenda = obs.get('hidden_agenda') or ''
    agenda_line = f'\nHidden mandate (confidential): {agenda[:200]}' if agenda else ''
    warn = '\nWARNING: Turn limit near — propose_consensus NOW.' if obs.get('turn_warning') else ''
    return (
        f'You are {agent_id} in a multi-phase medical negotiation.\n'
        f'Phase: {phase} (phase_turn={pturn})\n'
        f'Private information:\n{pinfo}\n'
        f'Recent conversation:\n{hist_str}'
        f'{agenda_line}{warn}\n'
        f'Available actions: {avail}\n'
        f'If phase_turn >= 2 and you have shared key info, use propose_consensus.\n'
        f'Respond ONLY with valid JSON (no markdown):\n'
        '{"agent_id": "' + agent_id + '", "action_type": "<action>", '
        '"content": "<plain English>", "reasoning": "<30+ word medical reasoning>"}'
    )


def _generate_action(obs: dict, agent_id: str) -> dict:
    prompt  = _format_prompt(obs, agent_id)
    inputs  = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
    avail   = obs.get('available_actions', VALID_ACTIONS)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 256,
            do_sample      = True,
            temperature    = 0.7,
            top_p          = 0.9,
            pad_token_id   = tokenizer.pad_token_id,
        )
    raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    # Parse JSON
    action = None
    try:
        action = json.loads(raw)
    except Exception:
        m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
        if m:
            try: action = json.loads(m.group())
            except Exception: pass
    if action is None or action.get('action_type') not in avail:
        # Fallback: pick best available action heuristically
        if 'propose_consensus' in avail and obs.get('phase_turn', 0) >= 2:
            atype = 'propose_consensus'
        elif 'accept_consensus' in avail:
            atype = 'accept_consensus'
        elif 'share_information' in avail:
            atype = 'share_information'
        else:
            atype = avail[0] if avail else 'request_clarification'
        action = {
            'agent_id':   agent_id,
            'action_type': atype,
            'content':    raw[:300] if raw else 'Sharing my clinical assessment.',
            'reasoning':  'Continuing negotiation based on available evidence and clinical guidelines.',
        }
    action['agent_id'] = agent_id
    return action, raw


def _api_post(path: str, payload: dict) -> dict | None:
    try:
        r = requests.post(f'{SPACE_URL}{path}', json=payload, timeout=TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'  API error {path}: {e}')
        return None


def _api_get(path: str) -> dict | None:
    try:
        r = requests.get(f'{SPACE_URL}{path}', timeout=TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'  API error {path}: {e}')
        return None

print('Helpers ready')

In [ ]:
# ============================================================
# SECTION 5: ROLLOUT EPISODE + REWARD FUNCTION
# ============================================================

def rollout_episode(task_id: str) -> tuple[float, list[str], list[str]]:
    """
    Runs one full episode against the Space API.
    Returns (final_score, prompts_list, completions_list).
    final_score from GET /state after done=True.
    On any Space error: returns (0.0, [], []).
    """
    state = _api_post('/reset', {'task_id': task_id})
    if state is None:
        return 0.0, [], []

    obs_a    = state.get('obs_agent_a', {})
    obs_b    = state.get('obs_agent_b', {})
    done     = False
    turn     = 0
    agent_id = 'agent_a'
    obs      = obs_a
    prompts, completions = [], []

    while not done and turn < MAX_TURNS:
        turn += 1
        action, raw = _generate_action(obs, agent_id)
        prompts.append(_format_prompt(obs, agent_id))
        completions.append(raw)

        step = _api_post('/step', {'action': action})
        if step is None:
            return 0.0, prompts, completions

        obs_a = step.get('obs_agent_a', obs_a)
        obs_b = step.get('obs_agent_b', obs_b)
        done  = step.get('done', False)

        agent_id = 'agent_b' if agent_id == 'agent_a' else 'agent_a'
        obs      = obs_b if agent_id == 'agent_b' else obs_a
        time.sleep(0.2)

    # Get final score
    full_state = _api_get('/state')
    if full_state is None:
        return 0.0, prompts, completions

    # episode_result embedded in last step, or pull from state
    ep_result = step.get('episode_result') if step else None
    if ep_result:
        score = float(ep_result.get('total_reward', 0.0))
    else:
        score = float(full_state.get('cumulative_reward', 0.0))

    score = max(0.0, min(1.0, score))
    return score, prompts, completions


# GRPO reward function — called by GRPOTrainer per completion batch
MEDICAL_KW = [
    'patient','diagnosis','treatment','troponin','ecg','sepsis','antibiotic',
    'heparin','naloxone','meningitis','embolism','clinical','prognosis',
    'vital','triage','dosage','imaging','labs','saturation',
]

def negotiation_reward_fn(prompts: list, completions: list, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        r = 0.0
        action = None
        try:
            action = json.loads(completion)
        except Exception:
            m = re.search(r'\{[^{}]+\}', completion, re.DOTALL)
            if m:
                try: action = json.loads(m.group())
                except Exception: pass
        if action is None:
            rewards.append(-0.5)
            continue
        atype     = action.get('action_type', '')
        content   = action.get('content', '')
        reasoning = action.get('reasoning', '')
        combined  = (content + ' ' + reasoning).lower()
        if atype not in VALID_ACTIONS:
            rewards.append(-0.3)
            continue
        med_hits = sum(1 for kw in MEDICAL_KW if kw in combined)
        if atype == 'share_information':
            r = 0.3 if med_hits >= 3 else (0.1 if med_hits >= 1 else 0.0)
        elif atype == 'propose_consensus':
            r = 0.4
        elif atype == 'accept_consensus':
            r = 0.5
        elif atype in ('flag_bias', 'flag_agenda'):
            r = 0.6
        elif atype == 'challenge_proposal':
            r = 0.2
        elif atype == 'request_clarification':
            r = 0.05
        if len(reasoning.split()) < 20:
            r -= 0.2
        rewards.append(max(-1.0, min(1.0, round(r, 4))))
    return rewards

print('Rollout + reward fn ready')

In [ ]:
# ============================================================
# SECTION 6: GRPO TRAINING LOOP
# ============================================================
from trl import GRPOTrainer, GRPOConfig
from datasets import Dataset

training_args = GRPOConfig(
    output_dir                  = './grpo_negotiation_out',
    num_train_epochs            = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = GROUP_SIZE,
    learning_rate               = 2e-5,
    max_completion_length       = 256,
    num_generations             = GROUP_SIZE,
    report_to                   = 'none',
    logging_steps               = 5,
    warmup_ratio                = 0.05,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
)

rewards_log = {task: [] for task in TRAIN_TASKS}   # {task_id: [score, ...]}
all_rewards = []

for epoch in range(1, N_EPOCHS + 1):
    print(f'\n=== EPOCH {epoch}/{N_EPOCHS} ===')
    epoch_prompts  = []
    epoch_responses = []

    for task_id in TRAIN_TASKS:
        print(f'  Task: {task_id}')
        for ep in range(EPISODES_PER_TASK):
            score, prompts, completions = rollout_episode(task_id)
            rewards_log[task_id].append(score)
            all_rewards.append(score)
            epoch_prompts.extend(prompts)
            epoch_responses.extend(completions)
            if (ep + 1) % 5 == 0:
                recent_avg = sum(rewards_log[task_id][-5:]) / 5
                print(f'    ep {ep+1:>2}/{EPISODES_PER_TASK}  score={score:.4f}  '
                      f'recent_avg={recent_avg:.4f}')

    if not epoch_prompts:
        print('  No data collected — skipping GRPO update')
        continue

    dataset = Dataset.from_dict({
        'prompt':     epoch_prompts,
        'completion': epoch_responses,
    })

    trainer = GRPOTrainer(
        model         = model,
        args          = training_args,
        reward_funcs  = negotiation_reward_fn,
        train_dataset = dataset,
        tokenizer     = tokenizer,
    )
    try:
        trainer.train()
        print(f'  GRPO update done (epoch {epoch})')
    except Exception as e:
        print(f'  GRPO update failed: {e}')

print('\nTraining complete')

In [ ]:
# ============================================================
# SECTION 7: PLOT REWARD CURVE
# ============================================================
fig, ax = plt.subplots(figsize=(12, 5))

colors = {'single-round-consensus': '#4A90D9',
          'adversarial-information': '#E74C3C',
          'opioid-overdose':         '#2ECC71'}

for task_id in TRAIN_TASKS:
    scores = rewards_log[task_id]
    if not scores:
        continue
    x = list(range(1, len(scores) + 1))
    # Smoothed line (rolling window=5)
    smoothed = []
    win = 5
    for i in range(len(scores)):
        w = scores[max(0, i - win + 1): i + 1]
        smoothed.append(sum(w) / len(w))
    ax.plot(x, smoothed, label=task_id, color=colors.get(task_id),
            linewidth=2, marker='o', markersize=3)
    ax.scatter(x, scores, alpha=0.2, color=colors.get(task_id), s=10)

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Episode Number', fontsize=13)
ax.set_ylabel('Episode Reward (final_score)', fontsize=13)
ax.set_title('GRPO Training — Social Agent Negotiation\nReward Curve per Task (smoothed window=5)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.1, 1.05)
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
print('reward_curve.png saved')
plt.show()

In [ ]:
# ============================================================
# SECTION 8: SUMMARY TABLE
# ============================================================
print('\n--- Per-Task Reward Summary ---')
print(f'{"Task":<30} {"Episodes":>9} {"Mean Reward":>12} {"First 10 Avg":>13} {"Last 10 Avg":>13}')
print('-' * 80)
for task_id in TRAIN_TASKS:
    sc = rewards_log[task_id]
    if not sc:
        print(f'{task_id:<30} {0:>9} {"N/A":>12}')
        continue
    first10 = sc[:10]
    last10  = sc[-10:]
    print(f'{task_id:<30} {len(sc):>9} {sum(sc)/len(sc):>12.4f} '
          f'{sum(first10)/len(first10):>13.4f} {sum(last10)/len(last10):>13.4f}')

overall_first = all_rewards[:30] if len(all_rewards) >= 30 else all_rewards
overall_last  = all_rewards[-30:] if len(all_rewards) >= 30 else all_rewards
print(f'\nOverall mean: {sum(all_rewards)/max(len(all_rewards),1):.4f}')
print(f'First-30 avg: {sum(overall_first)/len(overall_first):.4f}')
print(f'Last-30 avg:  {sum(overall_last)/len(overall_last):.4f}')

In [ ]:
# ============================================================
# SECTION 9: SAVE LoRA ADAPTER TO HUB
# ============================================================
if HF_TOKEN:
    print(f'Pushing LoRA adapter to {HF_REPO_ID} ...')
    try:
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f'Pushed: https://huggingface.co/{HF_REPO_ID}')
    except Exception as e:
        print(f'Push failed: {e}')
else:
    # Save locally as fallback
    model.save_pretrained('./grpo_lora_adapter')
    tokenizer.save_pretrained('./grpo_lora_adapter')
    print('Saved locally to ./grpo_lora_adapter (no HF_TOKEN)')

print('Training complete. Reward curve saved to reward_curve.png')